# ForgetProof — 02: Unlearn + audit

Base paper: Xiao et al., *Robust LLM Unlearning Against Relearning Attacks: The
Minor Components in Representations Matter* (arXiv:2605.11685) — MCU.

Runs 8 unlearning configurations against the target model, in 4 baseline/MCU
pairs:

| Baseline (paper) | + MCU |
|---|---|
| `npo` | `npo_mcu` **(our extension — the paper never applies MCU to an output-level loss)** |
| `rmu` | `rmu_mcu` (paper Eq 12) |
| `mlp_breaking` | `mlp_breaking_mcu` (paper Eq 14) |
| `grad_ascent`, `grad_diff` | — (kept as weaker reference baselines) |

Then runs our audit suite (RTT, membership inference, rephrase extraction,
representation probe) on every resulting model, and reproduces the paper's
own diagnostic (per-PC change ratio / recovery ratio, their Observations 2-3)
on OUR model+dataset for the NPO vs NPO-MCU pair, since that comparison is
the one the paper itself never ran.

Run 01_setup_and_baseline.ipynb first so the target model is saved to Drive.

In [ ]:
import os

# Absolute path + os.chdir (not relative os.path.isdir('forgetproof') + %cd) so this
# cell is idempotent: safe whether the kernel is fresh OR this is a second "Run all"
# in a kernel that's already sitting inside /content/forgetproof. The old relative-path
# version re-cloned into a nested forgetproof/forgetproof on a same-kernel rerun and
# left later cells unable to `import src` (ModuleNotFoundError: No module named 'src').
REPO_DIR = '/content/forgetproof'
if os.path.isdir(REPO_DIR):
    os.chdir(REPO_DIR)
    !git pull
else:
    !git clone https://github.com/vinay-reddaboina/forgetproof.git {REPO_DIR}
    os.chdir(REPO_DIR)
!pip install -q -r requirements.txt

from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import copy, torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from src.data_utils import load_tofu, load_tofu_real_authors

device = "cuda" if torch.cuda.is_available() else "cpu"
TARGET_DIR = "/content/drive/MyDrive/forgetproof/target_model"

tokenizer = AutoTokenizer.from_pretrained(TARGET_DIR)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token  # phi-1.5 has no pad token by default

# Deliberately kept on CPU: base_model is only ever a deepcopy source (each
# unlearn_* function moves its own copy to `device` internally) and, later,
# an input to the PCA-diagnostics cell -- it's never needed on GPU for
# training or auditing. Every unlearning method already keeps 1-2 full fp16
# model copies + optimizer state on the GPU at once, and a T4 (~14.56GB
# usable) has no room to spare for a third idle copy.
base_model = AutoModelForCausalLM.from_pretrained(TARGET_DIR, torch_dtype=torch.float16)

FORGET_SPLIT = "forget10"
forget_ds, retain_ds = load_tofu(FORGET_SPLIT)
unseen_ds = load_tofu_real_authors()
print("loaded target model + data (base_model kept on CPU -- see comment above)")


In [ ]:
# Optional: paraphrased forget-set questions for the rephrase-extraction attack.
# TOFU's forget10_perturbed split has these; falls back to None (attack skipped)
# if it's not available for this split in the installed datasets version.

paraphrased_examples = None
try:
    from datasets import load_dataset
    perturbed = load_dataset("locuslab/TOFU", "forget10_perturbed")["train"]
    paraphrased_examples = [
        {"paraphrased_question": ex["paraphrased_question"], "answer": ex["answer"]}
        for ex in perturbed.select(range(min(50, len(perturbed))))
    ]
    print("loaded", len(paraphrased_examples), "paraphrased examples")
except Exception as e:
    print("no perturbed split available, skipping rephrase attack:", e)

In [ ]:
# Group forget-set questions by fictitious author for the representation-probe attack.
# TOFU has no explicit author/id column, but each split is unshuffled blocks of
# 20 consecutive QA pairs per author (verified against the dataset) -- see
# src/data_utils.group_by_author_block. Must run on forget_ds BEFORE any shuffling.

from src.data_utils import group_by_author_block

entities_for_probe = group_by_author_block(forget_ds, block_size=20, max_authors=10, questions_per_author=10)
print("probe entities:", list(entities_for_probe.keys()))

In [ ]:
import os, sys, json, shutil
REPO_DIR = '/content/forgetproof'
if os.getcwd() != REPO_DIR and os.path.isdir(REPO_DIR):
    os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

from src.unlearn.grad_ascent import unlearn_grad_ascent
from src.unlearn.grad_diff import unlearn_grad_diff
from src.unlearn.npo import unlearn_npo
from src.unlearn.npo_mcu import unlearn_npo_mcu
from src.unlearn.rmu import unlearn_rmu
from src.unlearn.mlp_breaking import unlearn_mlp_breaking
from src.eval.run_eval import run_full_audit, save_results

# layer_idx=-1 (last transformer block's MLP) and mcu_k=8 (mid-range of the
# paper's own grid search {1,2,4,8,16,32,64}, Appendix F) are shared defaults
# for every MCU-based method below, so the baseline/MCU pairs differ in
# exactly one thing: whether the minor-component projection is applied.
LAYER_IDX = -1
MCU_K = 8

# Only npo/npo_mcu are needed afterward (the PCA-diagnostics cell below
# compares exactly that pair). Keeping all 8 fine-tuned 1.3B copies on the
# GPU at once would need ~20GB -- more than a T4's 16GB -- so every other
# method's model is freed right after its results are recorded.
KEEP_FOR_DIAGNOSTICS = {"npo", "npo_mcu"}

# --- Checkpointing to Drive ---------------------------------------------
# A free-tier T4 runtime can disconnect at any point (idle timeout or the
# session-length cap), and Colab's local disk is wiped when that happens.
# Previously this loop only wrote results to git in the FINAL cell, after
# all 8 methods finished -- so any mid-run disconnect lost every method
# trained so far, forcing a full restart from grad_ascent every time.
# Now each method's results (and, for npo/npo_mcu, the trained model
# weights themselves, since the PCA-diagnostics cell below needs the
# actual model in memory) are saved to Drive -- which survives a
# disconnect -- the moment that method finishes. On the next "Run all",
# any method already saved on Drive is loaded back instead of retrained,
# so a disconnect only costs the one method that was mid-flight.
DRIVE_RESULTS_DIR = "/content/drive/MyDrive/forgetproof/results_checkpoints"
CKPT_DIR = "/content/drive/MyDrive/forgetproof/checkpoints"
os.makedirs(DRIVE_RESULTS_DIR, exist_ok=True)
os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs("results", exist_ok=True)

METHODS = {
    "grad_ascent":       lambda m: unlearn_grad_ascent(m, tokenizer, forget_ds, device=device),
    "grad_diff":         lambda m: unlearn_grad_diff(m, tokenizer, forget_ds, retain_ds, device=device),
    "npo":               lambda m: unlearn_npo(m, tokenizer, forget_ds, retain_ds, device=device),
    "npo_mcu":           lambda m: unlearn_npo_mcu(m, tokenizer, forget_ds, retain_ds, layer_idx=LAYER_IDX, mcu_k=MCU_K, device=device),
    "rmu":               lambda m: unlearn_rmu(m, tokenizer, forget_ds, retain_ds, layer_idx=LAYER_IDX, device=device, use_mcu=False),
    "rmu_mcu":           lambda m: unlearn_rmu(m, tokenizer, forget_ds, retain_ds, layer_idx=LAYER_IDX, mcu_k=MCU_K, device=device, use_mcu=True),
    "mlp_breaking":      lambda m: unlearn_mlp_breaking(m, tokenizer, forget_ds, retain_ds, layer_idx=LAYER_IDX, device=device, use_mcu=False),
    "mlp_breaking_mcu":  lambda m: unlearn_mlp_breaking(m, tokenizer, forget_ds, retain_ds, layer_idx=LAYER_IDX, mcu_k=MCU_K, device=device, use_mcu=True),
}

unlearned_models = {}   # only npo/npo_mcu kept around, for the PCA-diagnostics cell below
all_results = {}

for name, unlearn_fn in METHODS.items():
    local_result_path = f"results/{name}.json"
    drive_result_path = f"{DRIVE_RESULTS_DIR}/{name}.json"
    ckpt_path = f"{CKPT_DIR}/{name}.pt"
    unlearned = None
    model_copy = None

    if os.path.exists(drive_result_path):
        print(f"\n=== {name} (already done in an earlier run -- loading from Drive, not retraining) ===")
        with open(drive_result_path) as f:
            results = json.load(f)
        shutil.copy(drive_result_path, local_result_path)  # so the final git push in the last cell still has it
        if name in KEEP_FOR_DIAGNOSTICS:
            unlearned = copy.deepcopy(base_model)
            unlearned.load_state_dict(torch.load(ckpt_path, map_location="cpu"))
            unlearned.to(device)
    else:
        print(f"\n=== {name} ===")
        model_copy = copy.deepcopy(base_model)
        unlearned = unlearn_fn(model_copy)

        results = run_full_audit(
            unlearned, tokenizer, method_name=name,
            forget_ds=forget_ds, retain_ds=retain_ds, unseen_ds=unseen_ds,
            paraphrased_examples=paraphrased_examples,
            entities_for_probe=entities_for_probe,
            device=device,
        )
        save_results(results, local_result_path)
        shutil.copy(local_result_path, drive_result_path)
        if name in KEEP_FOR_DIAGNOSTICS:
            torch.save(unlearned.state_dict(), ckpt_path)
        print(f"checkpointed {name} to Drive")

    all_results[name] = results

    if name in KEEP_FOR_DIAGNOSTICS:
        unlearned_models[name] = unlearned
    elif unlearned is not None:
        del unlearned
    if model_copy is not None:
        del model_copy
    torch.cuda.empty_cache()


In [ ]:
# Quick comparison table -- headline numbers per method
import pandas as pd

rows = []
for name, r in all_results.items():
    rows.append({
        "method": name,
        "RTT recovery": r["relearning_attack"]["recovery_score"],
        "MIA leak": r["membership_inference"]["leak_score"],
        "rephrase leak rate": r.get("rephrase_extraction", {}).get("leak_rate"),
        "probe acc above chance": r.get("representation_probe", {}).get("above_chance"),
        "model utility loss": r["model_utility_loss"],
    })
pd.DataFrame(rows)

In [ ]:
# PCA diagnostics -- reproduces the paper's own Observations 2 and 3 (their
# Figure 2b/2c) on OUR model, for the npo vs npo_mcu pair specifically, since
# that's the comparison the paper itself never ran (Section 4 only wires MCU
# into RMU and MLP Breaking). Confirms whether our extension actually shifts
# unlearning-induced change into the minor-component subspace like intended.

from src.unlearn.mcu import extract_principal_components
from src.eval.pca_diagnostics import (
    explained_variance_profile, change_ratio_per_pc, recovery_ratio_per_pc,
)
from src.attacks.relearning import relearning_attack
import json as _json

base_model.to(device)  # was kept on CPU during training (see cell 2) -- needed on GPU here

pca_stats = extract_principal_components(
    base_model, tokenizer, forget_ds, layer_idx=LAYER_IDX, K=32, device=device
)

diagnostics = {
    "explained_variance": explained_variance_profile(
        base_model, tokenizer, forget_ds, layer_idx=LAYER_IDX, max_pcs=32, device=device
    ),
}

for name in ["npo", "npo_mcu"]:
    print(f"diagnostics for {name}...")
    change_ratio = change_ratio_per_pc(
        base_model, unlearned_models[name], tokenizer, forget_ds, pca_stats,
        layer_idx=LAYER_IDX, device=device,
    )
    # re-run the RTT attack once more here (cheap -- a handful of steps) so we
    # get back the actual relearned model + its exact holdout split
    rtt = relearning_attack(
        unlearned_models[name], tokenizer, forget_ds, device=device, return_model=True
    )
    recovery_ratio = recovery_ratio_per_pc(
        base_model, unlearned_models[name], rtt["model"], tokenizer,
        rtt["holdout_ds"], pca_stats, layer_idx=LAYER_IDX, device=device,
    )
    diagnostics[name] = {"change_ratio_per_pc": change_ratio, "recovery_ratio_per_pc": recovery_ratio}

with open("results/pca_diagnostics.json", "w") as f:
    _json.dump(diagnostics, f, indent=2)
print("saved -> results/pca_diagnostics.json")

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(diagnostics["explained_variance"])
axes[0].set_title("Explained variance per PC (Observation 1)")
axes[0].set_xlabel("Principal component"); axes[0].set_ylabel("Variance ratio")

for name, style in [("npo", "--"), ("npo_mcu", "-")]:
    axes[1].plot(diagnostics[name]["change_ratio_per_pc"], style, label=name)
axes[1].set_title("Unlearning change ratio per PC (Observation 2)")
axes[1].set_xlabel("Principal component"); axes[1].legend()

for name, style in [("npo", "--"), ("npo_mcu", "-")]:
    axes[2].plot(diagnostics[name]["recovery_ratio_per_pc"], style, label=name)
axes[2].set_title("RTT recovery ratio per PC (Observation 3)")
axes[2].set_xlabel("Principal component"); axes[2].legend()

plt.tight_layout()
plt.savefig("results/pca_diagnostics.png", dpi=150)
plt.show()
print("If npo_mcu's change-ratio mass sits further right (higher-index PCs) than")
print("npo's, and its recovery-ratio curve stays lower, the extension is doing")
print("what it's supposed to: concentrating forgetting in minor components that")
print("resist the relearning attack -- matching the paper's story for RMU/MLP-Breaking.")

In [ ]:
# Push results json + diagnostics back to the repo
!git add results/*.json results/*.png
!git commit -m "Add audit results + PCA diagnostics for all 8 methods"
!git push